# C2 · The same model at different prevalences: the AUC stays, the precision collapses

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Measure what happens to the metrics of a **fixed** classifier when the fraction of positives in the test data drops from 50% to 0.5%. Derive the formula of the
precision as a function of recall, false positive rate and prevalence, and correct the probabilities for the change of prevalence.

## Dataset

`generate_test(prevalence)` (given) draws from a large test pool a sample with the desired prevalence (20,000 negatives and a proportional number of positives). The model `m`,
a logistic, is trained **once** on 3000 positives and 3000 negatives (prevalence 50%).

## Educational purpose

Understand why a model that "works" in the lab (balanced data) can be unusable in production (rare event): AUC, recall and FPR are properties of the model, the
precision is **not**. And understand how a probability is corrected when the prevalence changes.

## Task

1. For `prevalences = [0.5, 0.2, 0.05, 0.01, 0.005]` build `table` (index: prevalence) with: `auc`, `avg_precision`, `accuracy` (threshold 0.5), `baseline` (accuracy of someone who always says negative), `precision`, `recall`, `mean_p` (mean of the predicted probabilities), `logloss`.
2. From the false positive rate and the recall measured at prevalence 0.5 compute `theoretical_precision[π] = R·π / (R·π + FPR·(1−π))` for every prevalence (`pd.Series`, same index) and check that it matches the measured precision.
3. **Prior correction.** The probabilities of the model assume prevalence 0.5. For a prevalence `π` the odds get multiplied by $\frac{\pi/(1-\pi)}{0.5/0.5}$. Add to `table` the columns `corrected_mean_p` and `corrected_logloss` computed with the corrected probabilities.
4. After the correction threshold 0.5 means something else entirely: add `corrected_recall` (recall with threshold 0.5 on the corrected probabilities) and answer with `correction_cost = True` if at prevalence 0.005 the corrected recall is below 0.1.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, log_loss, precision_score, recall_score, roc_auc_score

X, y = make_classification(n_samples=80000, n_features=8, n_informative=5, n_redundant=0, class_sep=0.8, flip_y=0.02, random_state=0)
pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
m = LogisticRegression(max_iter=1000).fit(X[np.r_[pos[:3000], neg[:3000]]], y[np.r_[pos[:3000], neg[:3000]]])
pos_te, neg_te = pos[3000:], neg[3000:23000]                    # 20,000 test negatives


def generate_test(prevalence, seed=0):
    n_pos = int(round(prevalence / (1 - prevalence) * len(neg_te)))
    idx = np.r_[neg_te, np.random.default_rng(seed).choice(pos_te, n_pos, replace=False)]
    return X[idx], y[idx]


prevalences = [0.5, 0.2, 0.05, 0.01, 0.005]

## Your work

In [ ]:
rows = {}
for pi in prevalences:
    X_te, y_te = generate_test(pi)
    p = m.predict_proba(X_te)[:, 1]
    rows[pi] = ...
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
theoretical_precision = None

In [ ]:
correction_cost = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Correction: `odds = p / (1 - p); k = (pi / (1 - pi)) / 1.0; p_c = odds * k / (1 + odds * k)`. FPR at prevalence 0.5: `((p >= 0.5) & (y == 0)).sum() / (y == 0).sum()`. Precision: `R * pi / (R * pi + FPR * (1 - pi))`.'):
    check(list(table.columns[:8]) == ["auc", "avg_precision", "accuracy", "baseline", "precision", "recall", "mean_p", "logloss"], "first eight columns correct", "check names and order")
    check(table["auc"].max() - table["auc"].min() < 0.04 and table["recall"].max() - table["recall"].min() < 0.1, "AUC and recall are (almost) constant as the prevalence varies (they vary only by sampling noise)", "they are properties of the model")
    check(table.loc[0.01, "precision"] < 0.1 and table.loc[0.005, "precision"] < 0.05, "with prevalence 1% the precision is below 10%, with 0.5% below 5%", "precision at threshold 0.5")
    check((table.loc[[0.05, 0.01, 0.005], "accuracy"] < table.loc[[0.05, 0.01, 0.005], "baseline"]).all(), "below 5% prevalence the model is less accurate than someone who always says 'negative'", "accuracy against baseline")
    check(theoretical_precision is not None and close(theoretical_precision.reindex(table.index).to_numpy(), table["precision"].to_numpy(), atol=0.03), "the precision formula matches the measured one (gap < 0.03)", "R*pi / (R*pi + FPR*(1-pi)) with R and FPR measured at prevalence 0.5")
    check("corrected_mean_p" in table and close(table["corrected_mean_p"].to_numpy(), np.array(table.index), rtol=0.25), "after the correction the mean predicted probability is close to the prevalence (within 25%)", "odds * k / (1 + odds * k), with k = (pi/(1-pi))")
    check((table["corrected_logloss"].iloc[1:] < table["logloss"].iloc[1:]).all() and table.loc[0.005, "corrected_logloss"] < 0.1, "the log-loss improves at every prevalence below 50% (at 0.5% below 0.1)", "compare logloss and corrected_logloss")
    check(correction_cost is True and table.loc[0.005, "corrected_recall"] < 0.1 and table.loc[0.005, "corrected_recall"] < table.loc[0.005, "recall"], "threshold 0.5 on the corrected probabilities finds almost no positive: the threshold must be revised", "corrected_recall at prevalence 0.005")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_prevalence_and_precision.ipynb`. Open it after you pass the check, or after you have really tried.